In [1]:
pip install torch

In [2]:
data = [
    (["John", "lives", "in", "New", "York"], ["B-PER", "O", "O", "B-LOC", "I-LOC"]),
    (["Mary", "works", "at", "Google"], ["B-PER", "O", "O", "B-ORG"]),
    (["Amazon", "is", "in", "Seattle"], ["B-ORG", "O", "O", "B-LOC"])
]

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

word_to_ix = {}
tag_to_ix = {}
for sentence, tags in data:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)
    for tag in tags:
        if tag not in tag_to_ix:
            tag_to_ix[tag] = len(tag_to_ix)
ix_to_tag = {v:k for k,v in tag_to_ix.items()}



In [3]:
#to convert data to sequence
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

In [4]:
import torch.nn as nn

class LSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = torch.log_softmax(tag_space, dim=1)
        return tag_scores

In [5]:
EMBEDDING_DIM = 6
HIDDEN_DIM = 6

model = LSTMTagger(
    EMBEDDING_DIM,
    HIDDEN_DIM,
    len(word_to_ix),
    len(tag_to_ix)
)

loss_function = nn.NLLLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [6]:
for epoch in range(100):
    for sentence, tags in data:
        model.zero_grad()
        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)
        tag_scores = model(sentence_in)
        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()

In [7]:
with torch.no_grad():
    inputs = prepare_sequence(["Mary", "works", "at", "Google"], word_to_ix)
    tag_scores = model(inputs)
    predicted_tags = torch.argmax(tag_scores, dim=1)

    for word, tag_ix in zip(["Mary", "works", "at", "Google"], predicted_tags):
        print(word, ix_to_tag[tag_ix.item()])

Mary B-PER
works O
at O
Google B-ORG
